<a href="https://colab.research.google.com/github/TinkerTechie/FakeNewsDetector/blob/main/Copy_of_model1_for_fake_news_generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import json

# Load the notebook
with open('Copy_of_model1_for_fake_news_generator.ipynb', 'r') as f:
    nb = json.load(f)

# Remove or fix metadata.widgets in all cells
for cell in nb.get('cells', []):
    if 'metadata' in cell and 'widgets' in cell['metadata']:
        # Option A: Remove the widgets metadata entirely
        del cell['metadata']['widgets']

        # Option B: If you need widgets, add state key:
        # cell['metadata']['widgets'] = {"state": {}}

# Save the fixed notebook
with open('/content/politifact_final_dataset (1) - politifact_final_dataset (1).csv.csv', 'w') as f:
    json.dump(nb, f, indent=2)

print("✅ Notebook fixed!")

FileNotFoundError: [Errno 2] No such file or directory: 'Copy_of_model1_for_fake_news_generator.ipynb'

In [ ]:
!pip install requests beautifulsoup4 playwright pandas
!playwright install

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 MB 21.5 MB/s eta 0:00:00
(node:894) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVEs are not issued for `url.parse()` vulnerabilities.
(Use `node --trace-deprecation ...` to show where the warning was created)
167.3 MiB [] 0% 0.0s167.3 MiB [] 0% 21.0s167.3 MiB [] 0% 13.3s167.3 MiB [] 0% 9.9s167.3 MiB [] 1% 3.9s167.3 MiB [] 2% 3.6s167.3 MiB [] 3% 2.7s167.3 MiB [] 5% 2.2s167.3 MiB [] 6% 2.0s167.3 MiB [] 7% 1.8s167.3 MiB [] 8% 1.7s167.3 MiB [] 10% 1.6s167.3 MiB [] 11% 1.5s167.3 MiB [] 13% 1.4s167.3 MiB [] 14% 1.4s167.3 MiB [] 15% 1.4s167.3 MiB [] 16% 1.4s167.3 MiB [] 17% 1.3s167.3 MiB [] 18% 1.3s167.3 MiB [] 19% 1.3s167.3 MiB [] 20% 1.2s167.3 MiB [] 22% 1.2s167.3 MiB [] 23% 1.1s167.3 MiB [] 25% 1.1s167.3 MiB [] 26% 1.1s167.3 MiB [] 28% 1.0s167.3 MiB [] 30% 1.0s167.3 MiB [] 31% 1.0s167.3 MiB [] 32% 0.9s167.3 MiB [] 34% 0.9s1

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import signal
import os

# =========================
# PATHS
# =========================
INPUT_CSV = "/content/politifact_final_dataset (1) - politifact_final_dataset (1).csv.csv"
OUTPUT_CSV = "politifact_with_extracted_text.csv"

# =========================
# SETTINGS
# =========================
MIN_TEXT_LENGTH = 300
REQUEST_TIMEOUT = 5
READER_TIMEOUT = 8
ROW_TIMEOUT = 12   # max seconds per URL

BLOCKED_DOMAINS = [
    "linkedin.com","instagram.com","facebook.com",
    "twitter.com","x.com","docs.google.com","drive.google.com"
]

# =========================
# TIMEOUT HANDLER
# =========================
class TimeoutException(Exception):
    pass

def timeout_handler(signum, frame):
    raise TimeoutException()

signal.signal(signal.SIGALRM, timeout_handler)

# =========================
# HELPERS
# =========================
def normalize_url(url):
    if pd.isna(url):
        return None
    url = str(url).strip()
    if not url.startswith("http"):
        url = "https://" + url
    return url

def is_blocked(url):
    return any(d in url.lower() for d in BLOCKED_DOMAINS)

def clean_html(html):
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script","style","noscript","header","footer","nav"]):
        tag.extract()
    return soup.get_text(" ", strip=True)

# =========================
# EXTRACTION METHODS
# =========================
def try_requests(url):
    try:
        r = requests.get(url, headers={"User-Agent":"Mozilla/5.0"}, timeout=REQUEST_TIMEOUT)
        if r.status_code != 200:
            return None
        text = clean_html(r.text)
        if len(text) < MIN_TEXT_LENGTH:
            return None
        return text
    except:
        return None

def try_reader(url):
    try:
        r = requests.get(f"https://r.jina.ai/{url}", timeout=READER_TIMEOUT)
        if r.status_code != 200:
            return None
        text = r.text.strip()
        if len(text) < MIN_TEXT_LENGTH:
            return None
        return text
    except:
        return None

def extract_from_url(url):
    if not url or is_blocked(url):
        return None

    text = try_requests(url)
    if text:
        return text

    text = try_reader(url)
    if text:
        return text

    return None

# =========================
# SAFE WRAPPER WITH TIMEOUT
# =========================
def extract_with_timeout(url):
    try:
        signal.alarm(ROW_TIMEOUT)
        text = extract_from_url(url)
        signal.alarm(0)
        return text
    except TimeoutException:
        return None

# =========================
# LOAD DATASET
# =========================
df = pd.read_csv(INPUT_CSV)

extracted = []

# =========================
# MAIN LOOP
# =========================
for i, row in df.iterrows():
    print("Processing row:", i)

    # 1️⃣ dataset already has text
    if pd.notna(row.get("text")) and len(str(row["text"])) > MIN_TEXT_LENGTH:
        extracted.append(row["text"])
        continue

    # 2️⃣ skip known failed URLs
    if str(row.get("fetch_status")).lower() == "failed":
        extracted.append(None)
        continue

    # 3️⃣ normalize URL
    url = normalize_url(row.get("url"))

    # 4️⃣ extract safely
    article = extract_with_timeout(url)
    extracted.append(article)

# =========================
# SAVE RESULT
# =========================
df["extracted_text"] = extracted
output_dir = os.path.dirname(OUTPUT_CSV)
if output_dir: # Only create directory if it's not an empty string
    os.makedirs(output_dir, exist_ok=True)
df.to_csv(OUTPUT_CSV, index=False)
from google.colab import files
files.download("politifact_with_extracted_text.csv")

print("✅ DONE — saved to:", OUTPUT_CSV)

Processing row: 0
Processing row: 1
Processing row: 2
Processing row: 3
Processing row: 4
Processing row: 5
Processing row: 6
Processing row: 7
Processing row: 8
Processing row: 9
Processing row: 10
Processing row: 11
Processing row: 12
Processing row: 13
Processing row: 14
Processing row: 15
Processing row: 16
Processing row: 17
Processing row: 18
Processing row: 19
Processing row: 20
Processing row: 21
Processing row: 22
Processing row: 23
Processing row: 24
Processing row: 25
Processing row: 26
Processing row: 27
Processing row: 28
Processing row: 29
Processing row: 30
Processing row: 31
Processing row: 32
Processing row: 33
Processing row: 34
Processing row: 35
Processing row: 36
Processing row: 37
Processing row: 38
Processing row: 39
Processing row: 40
Processing row: 41
Processing row: 42
Processing row: 43
Processing row: 44
Processing row: 45
Processing row: 46
Processing row: 47
Processing row: 48
Processing row: 49
Processing row: 50
Processing row: 51
Processing row: 52
Pro

/tmp/ipython-input-795/117637169.py:52: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html, "html.parser")


Processing row: 953
Processing row: 954
Processing row: 955
Processing row: 956
Processing row: 957
Processing row: 958
Processing row: 959
Processing row: 960
Processing row: 961
Processing row: 962
Processing row: 963
Processing row: 964
Processing row: 965
Processing row: 966
Processing row: 967
Processing row: 968
Processing row: 969
Processing row: 970
Processing row: 971
Processing row: 972
Processing row: 973
Processing row: 974
Processing row: 975
Processing row: 976
Processing row: 977
Processing row: 978
Processing row: 979
Processing row: 980
Processing row: 981
Processing row: 982
Processing row: 983
Processing row: 984
Processing row: 985
Processing row: 986
Processing row: 987
Processing row: 988
Processing row: 989
Processing row: 990
Processing row: 991
Processing row: 992
Processing row: 993
Processing row: 994
Processing row: 995
Processing row: 996
Processing row: 997
Processing row: 998
Processing row: 999
Processing row: 1000
Processing row: 1001
Processing row: 10

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ DONE — saved to: politifact_with_extracted_text.csv


In [ ]:
df["extracted_text"].isnull().sum()

np.int64(399)

In [ ]:
final_text = df.loc[row.name, "extracted_text"]

In [ ]:
final_text = row["text"]
final_text

nan

In [ ]:
df = df.copy()

df["final_text"] = df["extracted_text"].fillna(df["text"])
df["final_text"] = df["final_text"].fillna("").astype(str)

df["url"] = df["url"].fillna("").astype(str)
df["label"] = df["label"].astype(int)

**Matrix 1 (Linguistic Style)**

Install sentiment library

In [ ]:
!pip install vaderSentiment

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 4.2 MB/s eta 0:00:00


In [ ]:
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

# =========================
# LEXICONS (edit/expand anytime)
# =========================
SENSATIONAL_WORDS = {
    "shocking","unbelievable","explosive","disturbing",
    "exposed","bombshell","outrageous","terrifying"
}

CLICKBAIT_PHRASES = [
    "you won't believe",
    "they don't want you to know",
    "secret",
    "hidden truth",
    "what happens next",
    "this will shock you"
]

HYPERBOLE_WORDS = {
    "always","never","everyone","everything",
    "guaranteed","forever","completely","absolutely"
}

# =========================
# TOKENIZER
# =========================
def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

# =========================
# FEATURE FUNCTIONS
# =========================
def sensational_density(text):
    words = tokenize(text)
    if not words:
        return 0
    count = sum(1 for w in words if w in SENSATIONAL_WORDS)
    return count / len(words)

def clickbait_density(text):
    text_lower = text.lower()
    word_count = max(len(tokenize(text)), 1)
    count = sum(text_lower.count(p) for p in CLICKBAIT_PHRASES)
    return count / word_count

def hyperbole_density(text):
    words = tokenize(text)
    if not words:
        return 0
    count = sum(1 for w in words if w in HYPERBOLE_WORDS)
    return count / len(words)

def caps_punct_intensity(text):
    words = text.split()
    if not words:
        return 0

    caps = sum(1 for w in words if w.isupper() and len(w) > 2)
    exclam = text.count("!")
    ques = text.count("?")

    return (caps + exclam + ques) / len(words)

def emotional_intensity(text):
    score = analyzer.polarity_scores(text)
    return abs(score["compound"])

# =========================
# APPLY MATRIX 1
# =========================
df["sensational_density"] = df["final_text"].apply(sensational_density)
df["clickbait_density"] = df["final_text"].apply(clickbait_density)
df["hyperbole_density"] = df["final_text"].apply(hyperbole_density)
df["caps_punct_intensity"] = df["final_text"].apply(caps_punct_intensity)
df["emotional_intensity"] = df["final_text"].apply(emotional_intensity)

print("✅ Matrix 1 features created")

✅ Matrix 1 features created


In [ ]:
df[[
    "sensational_density",
    "clickbait_density",
    "hyperbole_density",
    "caps_punct_intensity",
    "emotional_intensity"
]].describe()

,sensational_density,clickbait_density,hyperbole_density,caps_punct_intensity,emotional_intensity
count,1056.000000,1056.000000,1056.000000,1056.000000,1056.000000
mean,0.000058,0.000288,0.000516,0.033183,0.508895
std,0.000369,0.001811,0.001492,0.060599,0.447004
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.006954,0.624900
75%,0.000000,0.000000,0.000000,0.041667,0.983325
max,0.005195,0.039948,0.018987,0.832258,1.000000


In [ ]:
df.to_csv("/mnt/data/politifact_matrix1.csv", index=False)

**Matrix 2**

Install embedding model

In [ ]:
!pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import re

model = SentenceTransformer("all-MiniLM-L6-v2")

# =========================
# Helpers
# =========================
def split_headline_body(text):
    """
    Approximate headline as first sentence.
    Body = remaining text.
    """
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    if len(sentences) == 0:
        return "", ""
    if len(sentences) == 1:
        return sentences[0], sentences[0]
    return sentences[0], " ".join(sentences[1:])

def paragraph_split(text):
    """
    Split into pseudo-paragraphs.
    """
    paras = re.split(r"\n+|\.\s+", text)
    paras = [p.strip() for p in paras if len(p.strip()) > 40]
    return paras


# =========================
# Feature 1 — Headline-Body Similarity
# =========================
def headline_body_similarity(text):
    if not text or len(text) < 50:
        return 0

    headline, body = split_headline_body(text)

    emb = model.encode([headline, body])
    sim = cosine_similarity([emb[0]], [emb[1]])[0][0]

    return float(sim)


# =========================
# Feature 2 — Topic Drift
# =========================
def topic_drift(text):
    paras = paragraph_split(text)

    if len(paras) < 2:
        return 0

    emb = model.encode(paras)

    sims = []
    for i in range(len(emb) - 1):
        s = cosine_similarity([emb[i]], [emb[i+1]])[0][0]
        sims.append(s)

    # drift = variation in similarity
    return float(np.std(sims))


# =========================
# Apply Matrix 2
# =========================
df["headline_body_similarity"] = df["final_text"].apply(headline_body_similarity)
df["topic_drift"] = df["final_text"].apply(topic_drift)

print("✅ Matrix 2 features created")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Matrix 2 features created


In [ ]:
df[["headline_body_similarity","topic_drift"]].describe()

,headline_body_similarity,topic_drift
count,1056.000000,1056.000000
mean,0.303358,0.102510
std,0.293700,0.095193
min,-0.043679,0.000000
25%,0.000000,0.000000
50%,0.288241,0.122678
75%,0.554852,0.168967
max,1.000000,0.390140


In [ ]:
df.to_csv("/mnt/data/politifact_matrix2.csv", index=False)

**Matrix 3**

In [ ]:
from urllib.parse import urlparse
import numpy as np

# =========================
# Extract domain from URL
# =========================
def extract_domain(url):
    if not isinstance(url, str) or url.strip() == "":
        return "unknown"
    try:
        domain = urlparse(url).netloc.lower()
        if domain.startswith("www."):
            domain = domain[4:]
        return domain if domain else "unknown"
    except:
        return "unknown"

df["domain"] = df["url"].apply(extract_domain)

# =========================
# Compute fake ratio per domain
# =========================
domain_stats = df.groupby("domain")["label"].agg(["count","mean"]).reset_index()

domain_stats.columns = ["domain","article_count","fake_ratio"]

# =========================
# Convert to reliability
# =========================
domain_stats["source_reliability"] = 1 - domain_stats["fake_ratio"]

# =========================
# Merge back to dataset
# =========================
df = df.merge(
    domain_stats[["domain","source_reliability"]],
    on="domain",
    how="left"
)

# =========================
# Handle unseen / rare domains
# =========================
MIN_DOMAIN_ARTICLES = 3

rare_domains = domain_stats[
    domain_stats["article_count"] < MIN_DOMAIN_ARTICLES
]["domain"]

df.loc[df["domain"].isin(rare_domains), "source_reliability"] = 0.5
df["source_reliability"] = df["source_reliability"].fillna(0.5)

print("✅ Matrix 3 feature created")

✅ Matrix 3 feature created


In [ ]:
df["source_reliability"].describe()

,source_reliability
count,1056.000000
mean,0.625473
std,0.246766
min,0.000000
25%,0.500000
50%,0.500000
75%,0.848901
max,1.000000


In [ ]:
df.to_csv("/mnt/data/politifact_matrix3.csv", index=False)

In [ ]:
features = [
    "sensational_density",
    "clickbait_density",
    "hyperbole_density",
    "caps_punct_intensity",
    "emotional_intensity",
    "headline_body_similarity",
    "topic_drift",
    "source_reliability"
]

X = df[features]
y = df["label"]

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [ ]:
features = [
    "sensational_density",
    "clickbait_density",
    "hyperbole_density",
    "caps_punct_intensity",
    "emotional_intensity",
    "headline_body_similarity",
    "topic_drift",
    "source_reliability"
]

X = df[features]
y = df["label"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
model = LogisticRegression(max_iter=200)
model.fit(X_train_scaled, y_train)

LogisticRegression(max_iter=200)

In [ ]:
y_pred = model.predict(X_test_scaled)

In [ ]:
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy :", acc)
print("Precision:", prec)
print("Recall   :", rec)
print("F1 Score :", f1)

Accuracy : 0.7971698113207547
Precision: 0.7340425531914894
Recall   : 0.7931034482758621
F1 Score : 0.7624309392265194


In [ ]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.85      0.80      0.82       125
           1       0.73      0.79      0.76        87

    accuracy                           0.80       212
   macro avg       0.79      0.80      0.79       212
weighted avg       0.80      0.80      0.80       212



In [ ]:
import pandas as pd

importance = pd.Series(
    model.coef_[0],
    index=features
).sort_values(ascending=False)

print("Feature Importance:")
print(importance)

Feature Importance:
hyperbole_density           0.434729
sensational_density         0.255914
topic_drift                 0.240096
headline_body_similarity    0.175913
clickbait_density          -0.124320
caps_punct_intensity       -0.295358
emotional_intensity        -0.433528
source_reliability         -1.638400
dtype: float64


In [ ]:
df["fake_probability"] = model.predict_proba(
    scaler.transform(df[features])
)[:,1]

In [ ]:
df.to_csv("/mnt/data/politifact_with_matrices_and_model.csv", index=False)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    random_state=42
)

rf_model.fit(X_train_scaled, y_train)

y_pred_rf = rf_model.predict(X_test_scaled)

acc_rf = accuracy_score(y_test, y_pred_rf)
prec_rf = precision_score(y_test, y_pred_rf)
rec_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)

print("Random Forest Results")
print("Accuracy :", acc_rf)
print("Precision:", prec_rf)
print("Recall   :", rec_rf)
print("F1 Score :", f1_rf)

Random Forest Results
Accuracy : 0.8018867924528302
Precision: 0.7368421052631579
Recall   : 0.8045977011494253
F1 Score : 0.7692307692307693


In [ ]:
rf_importance = pd.Series(
    rf_model.feature_importances_,
    index=features
).sort_values(ascending=False)

print(rf_importance)

source_reliability          0.495967
caps_punct_intensity        0.106171
emotional_intensity         0.099720
hyperbole_density           0.092704
topic_drift                 0.084381
headline_body_similarity    0.067224
clickbait_density           0.026967
sensational_density         0.026867
dtype: float64
